# Customer Churn Prediction using Machine Learning

**Author**: Senior Machine Learning Engineer  
**Domain**: Customer Retention & Subscription Analytics  
**ML Type**: Supervised Learning (Binary Classification)  
**Primary Model**: Logistic Regression | **Compared Models**: Decision Tree, Random Forest, K-Nearest Neighbors, Support Vector Machine

---

## Executive Summary & Workflow Overview
Customer churn is a critical business metric representing the percentage of customers who discontinue their service. This enterprise-grade notebook implements an end-to-end Machine Learning pipeline to identify at-risk customers early, quantify their churn probability into discrete risk tiers, and provide actionable business recommendations.

--- 
## STEP 1: Import Required Libraries

In [ ]:
# Step 1: Import Core Data Science, ML & Visualization Libraries
import os
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Pipeline & Estimators
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
print('[INFO] All libraries imported successfully.')

--- 
## STEP 2: Load and Inspect Dataset

In [ ]:
# Step 2: Load dataset from CSV
data_path = os.path.join('..', 'dataset', 'customer_churn.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('dataset', 'customer_churn.csv')

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

print("=== FIRST 5 ROWS ===")
display(df.head())

print("\n=== LAST 5 ROWS ===")
display(df.tail())

print("\n=== DATASET COLUMNS ===")
print(df.columns.tolist())

print("\n=== DATASET INFO ===")
df.info()

print("\n=== NUMERICAL DESCRIPTIVE STATISTICS ===")
display(df.describe().T)

print("\n=== MISSING VALUES PER COLUMN ===")
print(df.isnull().sum())

print(f"\nDuplicate Rows Count: {df.duplicated().sum()}")

--- 
## STEP 3: Data Cleaning & Preprocessing

In [ ]:
# Step 3: Handle Duplicate Rows & Missing Values
df_clean = df.copy()

# 1. Drop Duplicates
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"[CLEANING] Removed {initial_len - len(df_clean)} duplicate row(s).")

# 2. Convert Data Types & Impute Missing TotalCharges
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
missing_count = df_clean['TotalCharges'].isnull().sum()
if missing_count > 0:
    imputed = df_clean['Tenure'] * df_clean['MonthlyCharges']
    df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(imputed)
    print(f"[CLEANING] Imputed {missing_count} missing value(s) in TotalCharges with (Tenure * MonthlyCharges).")

print(f"Cleaned Dataset Final Shape: {df_clean.shape}")

--- 
## STEP 4: Exploratory Data Analysis (EDA)

In [ ]:
# Step 4: Visualizations for Demographics, Usage & Churn
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Churn Class Distribution
sns.countplot(data=df_clean, x='Churn', ax=axes[0,0], palette=['#2ecc71', '#e74c3c'], hue='Churn', legend=False)
axes[0,0].set_title('Customer Churn Class Distribution', fontweight='bold')
axes[0,0].set_xlabel('Churn Status')
axes[0,0].set_ylabel('Count')
axes[0,0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. Gender Distribution by Churn
sns.countplot(data=df_clean, x='Gender', hue='Churn', ax=axes[0,1], palette=['#3498db', '#e74c3c'])
axes[0,1].set_title('Gender Distribution by Churn Status', fontweight='bold')
axes[0,1].grid(axis='y', linestyle='--', alpha=0.7)

# 3. Contract Type Distribution
sns.countplot(data=df_clean, x='Contract', hue='Churn', ax=axes[1,0], palette=['#9b59b6', '#e74c3c'])
axes[1,0].set_title('Contract Type vs Churn', fontweight='bold')
axes[1,0].grid(axis='y', linestyle='--', alpha=0.7)

# 4. Internet Service Distribution
sns.countplot(data=df_clean, x='InternetService', hue='Churn', ax=axes[1,1], palette=['#34495e', '#e74c3c'])
axes[1,1].set_title('Internet Service Type vs Churn', fontweight='bold')
axes[1,1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Continuous Variables Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df_clean, x='MonthlyCharges', hue='Churn', kde=True, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Monthly Charges Distribution by Churn', fontweight='bold')

sns.histplot(data=df_clean, x='Tenure', hue='Churn', kde=True, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Tenure (Months) Distribution by Churn', fontweight='bold')

sns.histplot(data=df_clean, x='Age', hue='Churn', kde=True, ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Age Distribution by Churn', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 7))
num_cols = df_clean.select_dtypes(include=[np.number])
corr = num_cols.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Numerical Feature Correlation Heatmap', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

--- 
## STEP 5: Feature Engineering & Encoding

In [ ]:
# Step 5: Convert Categorical Features via pd.get_dummies() and Map Target
df_prep = df_clean.drop(columns=['CustomerID'])
y = df_prep['Churn'].map({'Yes': 1, 'No': 0})
X_raw = df_prep.drop(columns=['Churn'])

# Apply One-Hot Encoding
cat_cols = X_raw.select_dtypes(include=['object', 'category', 'str']).columns
X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)

# Convert boolean to int
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

print(f"Processed Features Matrix X Shape: {X.shape}")
print(f"Target Vector y Distribution:\n{y.value_counts(normalize=True)}")

--- 
## STEP 6: Train / Test Split

In [ ]:
# Step 6: 80/20 Stratified Split with random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape : {X_test.shape}, y_test shape : {y_test.shape}")

--- 
## STEP 7 & 8: Pipeline Construction & Baseline Model Training

In [ ]:
# Step 7 & 8: Build Scikit-Learn Pipeline (StandardScaler + Logistic Regression)
pipeline_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Fit Model
pipeline_lr.fit(X_train, y_train)

# Predict Labels & Probabilities
y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]
print('[INFO] Baseline Logistic Regression pipeline trained successfully.')

--- 
## STEP 9: Primary Model Performance Metrics

In [ ]:
# Step 9: Compute Classification Evaluation Metrics
acc = accuracy_score(y_test, y_pred_lr)
prec = precision_score(y_test, y_pred_lr)
rec = recall_score(y_test, y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)
auc = roc_auc_score(y_test, y_prob_lr)

print("=" * 45)
print("     PRIMARY MODEL PERFORMANCE (LOGISTIC REGRESSION)")
print("=" * 45)
print(f" Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
print(f" Precision : {prec:.4f}")
print(f" Recall    : {rec:.4f}")
print(f" F1 Score  : {f1:.4f}")
print(f" ROC-AUC   : {auc:.4f}")
print("=" * 45 + "\n")

print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_lr, target_names=['Retained (No)', 'Churned (Yes)']))

--- 
## STEP 10: Performance Visualizations

In [ ]:
# Step 10: Generate Plots (Confusion Matrix, ROC Curve, PR Curve)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
axes[1].plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'Logistic Regression (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='#7f8c8d', linestyle='--')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--')

# 3. Precision-Recall Curve
precision_pts, recall_pts, _ = precision_recall_curve(y_test, y_prob_lr)
axes[2].plot(recall_pts, precision_pts, color='#8e44ad', lw=2.5)
axes[2].set_title('Precision-Recall Curve', fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].grid(True, linestyle='--')

plt.tight_layout()
plt.show()

--- 
## STEP 11: Algorithm Comparison & Cross-Validation

In [ ]:
# Step 11: Compare 5 Classifiers with Cross-Validation
algorithms = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier(max_depth=6, random_state=42))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'K-Nearest Neighbors': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=7))]),
    'Support Vector Machine': Pipeline([('scaler', StandardScaler()), ('model', SVC(probability=True, random_state=42))])
}

results = []
for name, pipe in algorithms.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    t1 = time.time()
    
    y_p = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else None
    
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1').mean()
    
    results.append({
        'Algorithm': name,
        'Train Acc': round(pipe.score(X_train, y_train), 4),
        'Test Acc': round(accuracy_score(y_test, y_p), 4),
        'Precision': round(precision_score(y_test, y_p, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_p, zero_division=0), 4),
        'F1 Score': round(f1_score(y_test, y_p, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan,
        '5-Fold CV F1': round(cv_f1, 4),
        'Exec Time (s)': round(t1 - t0, 3)
    })

df_res = pd.DataFrame(results)
display(df_res)

--- 
## BONUS FEATURE: Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Bonus: GridSearchCV Hyperparameter Tuning for Logistic Regression
param_grid = {
    'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'model__solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(algorithms['Logistic Regression'], param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_pipe = grid.best_estimator_
print(f"Best Parameters: {grid.best_params_}")
print(f"Best Cross-Validation ROC-AUC: {grid.best_score_:.4f}")

--- 
## STEP 12: Save Champion Model Artifact

In [ ]:
# Step 12: Export Model Artifact using Joblib
save_dir = os.path.join('..', 'saved_model')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'churn_prediction_model.pkl')

artifact = {
    'model_pipeline': best_pipe,
    'feature_names': X.columns.tolist(),
    'model_name': 'Tuned Logistic Regression'
}
joblib.dump(artifact, save_path)
print(f"Champion model exported successfully to '{save_path}'.")

--- 
## STEP 13 & 14: Real-time Customer Prediction Engine & Metric Explanations

In [ ]:
# Step 13: Interactive Prediction Function with Risk Tiers
def predict_customer_churn(customer_dict):
    artifact = joblib.load(save_path)
    pipeline = artifact['model_pipeline']
    feature_names = artifact['feature_names']
    
    # Standardize input
    df_in = pd.DataFrame([customer_dict])
    df_in['SeniorCitizen'] = 1 if customer_dict.get('Age', 30) >= 65 else 0
    df_in['TotalCharges'] = customer_dict.get('Tenure', 1) * customer_dict.get('MonthlyCharges', 50.0)
    
    cat_cols = df_in.select_dtypes(include=['object', 'category', 'str']).columns
    df_enc = pd.get_dummies(df_in, columns=cat_cols, drop_first=True)
    df_enc[df_enc.select_dtypes(include=['bool']).columns] = df_enc.select_dtypes(include=['bool']).astype(int)
    
    df_aligned = df_enc.reindex(columns=feature_names, fill_value=0)
    
    prob = pipeline.predict_proba(df_aligned)[0][1]
    pred = 'WILL CHURN' if prob >= 0.5 else 'WILL NOT CHURN'
    
    if prob < 0.25:
        risk, action = 'Low Risk', 'Maintain regular touchpoints.'
    elif prob < 0.50:
        risk, action = 'Medium Risk', 'Send proactive feature usage tips & loyalty newsletter.'
    elif prob < 0.75:
        risk, action = 'High Risk', 'Offer 15% contract upgrade discount & priority support.'
    else:
        risk, action = 'Very High Risk', 'Assign dedicated retention specialist immediately.'
        
    filled = int(round(25 * prob))
    bar = '[' + '=' * filled + '-' * (25 - filled) + f'] {prob*100:.1f}%'
    
    print("=" * 55)
    print(f" PREDICTION TARGET : {pred}")
    print(f" CHURN PROBABILITY : {bar}")
    print(f" RISK TIER         : {risk}")
    print(f" ACTION REQUIRED   : {action}")
    print("=" * 55)

# Test Sample
sample_customer = {
    'Age': 58, 'Gender': 'Female', 'Tenure': 4, 'MonthlyCharges': 98.0,
    'Contract': 'Month-to-month', 'InternetService': 'Fiber optic',
    'TechSupport': 'No', 'Complaints': 1, 'SupportTickets': 3,
    'SatisfactionScore': 1, 'UsageHours': 35.0
}
predict_customer_churn(sample_customer)

### Evaluation Metrics Reference Guide
- **Accuracy**: Ratio of correct predictions out of total predictions: $(TP + TN) / (TP + TN + FP + FN)$.
- **Precision**: Proportion of true positive churners among all predicted churners: $TP / (TP + FP)$. Reduces false alarms.
- **Recall**: Proportion of true positive churners correctly identified among all actual churners: $TP / (TP + FN)$. Reduces missed revenue loss.
- **F1 Score**: Harmonic mean of Precision and Recall: $2 \cdot (Precision \cdot Recall) / (Precision + Recall)$.
- **ROC-AUC Score**: Area under the True Positive Rate vs False Positive Rate curve across all decision thresholds.
- **Confusion Matrix**: Tabular representation of TP, FP, TN, and FN breakdown.